# Weather forecasts from OpenWeatherMap API

API documentation: https://openweathermap.org/api/forecast5?collection=current_forecast 

## Libraries and Constants

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
openweather_key = os.getenv("OPENWEATHER_KEY")

In [ ]:
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

## One city, one forecast

Start small by calling the API on a single location and exploring the returned data for one time

In [ ]:
# select one row of cities_df
berlin = cities_df.loc[0]

url = "https://api.openweathermap.org/data/2.5/forecast"
params = {"lat": berlin["latitude"], "lon" :berlin["longitude"], "appid":openweather_key}

response = requests.get(url, params=params)
response

A response code of `200` means a successful request! Let's start exploring.

In [ ]:
# transform output, check keys
response_json = response.json()
response_json.keys()

Let's go through the keys one at time to understand the data structure

In [ ]:
# looks like the status code
response_json['cod']

In [ ]:
# ???
response_json['message']

In [ ]:
# one forecast every 3 hours means 8 per day; five days times 8 makes 40 forecasts
response_json['cnt']

In [ ]:
# looks like the forecasts themselves
response_json['list']

In [ ]:
# matches 'cnt' above!
len(response_json['list'])

In [ ]:
# we have this info in the 'cities' table
response_json['city']

Time to explore one forecast and see what to keep.

In [ ]:
forecast = response_json['list'][0]
forecast

* dt looks like some kind of timestamp (use units=metric instead)
* main
    * temp looks crazy! 292 degrees!
    * feels_like might be more important for people than the actual temperature
    * temp_min/max might be worth keeping? maybe not
    * humidity might affect if people rent scooters or not
* weather looks like the kind of descriptions a weather app would show
* clouds probably not needed
* wind
    * speed or gust could be important
* visibility probably not needed
* pop 'probability of precipitation' seems useful
* rain and snow seem useful
* sys probably not needed
* dt_txt a much better timestamp

In [ ]:
forecast_data = {
    "temp": forecast['main']['temp'],
    "feels_like": forecast['main']['feels_like'],
    "humidity_%": forecast['main']['humidity'],
    "wind_speed": forecast['wind']['speed'],
    "wind_gust": forecast['wind']['gust'],
    "precipitation_%": forecast['pop'],
    "rain_3h": forecast['rain']['3h'],
    # docs say there's 'snow', but it's not here!
    "forecast_time": forecast['dt_txt']
}
forecast_data

## One city, all forecasts

Now that we have useful information from one forecast, let's try to extract the same information from the other 39, too.

In [ ]:
forecasts = []
for forecast in response_json['list']:
    forecast_data = {
        "temp": forecast['main']['temp'],
        "feels_like": forecast['main']['feels_like'],
        "humidity_%": forecast['main']['humidity'],
        "wind_speed": forecast['wind']['speed'],
        "wind_gust": forecast['wind']['gust'],
        "precipitation_%": forecast['pop'],
        "rain_3h": forecast['rain']['3h'],
        # docs say there's 'snow', but it's not here!
        "forecast_time": forecast['dt_txt']
    }
    forecasts.append(forecast_data)
forecasts

Oh no! One of the forecasts has no 'rain'!

In [ ]:
forecast # the iteration variable stays stuck wherever the loop broke

Indeed, there's no 'rain' in this dictionary, just like we saw no 'snow' before. JSON documents are semi-structured; where there's no data, they might leave things out rather than listing a null value. Thankfully, python dictionaries have a method that allow us to retrieve a key that might not be there: `.get()`. We can even set the value to return if the key isn't found.

In [ ]:
forecast.get('dt_txt')

In [ ]:
forecast.get('rain', {}) # {} is the default value

By returning an empty dictionary as the default, we can chain more searches together.

In [ ]:
# returns a reasonable value for "no rain in the last three hours"
forecast.get('rain', {}).get('3h', 0)

In [ ]:
# works just fine if there has been rain
response_json['list'][0].get('rain', {}).get('3h', 0)

We'll adjust our loop to use `.get()`

In [ ]:
forecasts = []
for forecast in response_json['list']:
    forecast_data = {
        "temp": forecast['main']['temp'],
        "feels_like": forecast['main']['feels_like'],
        "humidity_%": forecast['main']['humidity'],
        "wind_speed": forecast['wind']['speed'],
        "wind_gust": forecast['wind']['gust'],
        "precipitation_%": forecast['pop'],
        "rain_3h": forecast.get('rain', {}).get('3h', 0),
        "snow_3h": forecast.get('snow', {}).get('3h', 0),
        "forecast_time": forecast['dt_txt']
    }
    forecasts.append(forecast_data)
forecasts

In [ ]:
pd.DataFrame(forecasts)

## All cities, all forecasts

Now let's expand one more time by adding a loop to handle each of our three cities.

In [ ]:
forecasts = []
url = "https://api.openweathermap.org/data/2.5/forecast"

# loop through cities_df
for _, row in cities_df.iterrows():
    # don't forget to add a parameter to use metric units!
    params = {"lat": row["latitude"], "lon" :row["longitude"], "appid":openweather_key, "units": "metric"}
    response = requests.get(url, params=params)
    response_json = response.json()
    
    for forecast in response_json['list']:
        forecast_data = {
            "temp": forecast['main']['temp'],
            "feels_like": forecast['main']['feels_like'],
            "humidity_%": forecast['main']['humidity'],
            "wind_speed": forecast['wind']['speed'],
            "wind_gust": forecast['wind']['gust'],
            "precipitation_%": forecast['pop'],
            "rain_3h": forecast.get('rain', {}).get('3h', 0),
            "snow_3h": forecast.get('snow', {}).get('3h', 0),
            "forecast_time": forecast['dt_txt'],
            "city_id": row["city_id"] # add city_id to reference back to 'cities' table
        }
        forecasts.append(forecast_data)

forecast_df = pd.DataFrame(forecasts)
forecast_df

In [ ]:
# useful for writing the SQL table definition
forecast_df.info()